In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
.appName("Retail Sales Analysis") \
.getOrCreate()

sales_data = [
("T001","North","Delhi","Store-01","Laptop","2024-01-01",75000),
("T002","North","Delhi","Store-01","Mobile","2024-01-02",32000),
("T003","North","Chandigarh","Store-02","Tablet","2024-01-03",26000),
("T004","South","Bangalore","Store-03","Laptop","2024-01-01",78000),
("T005","South","Chennai","Store-04","Mobile","2024-01-02",30000),
("T006","South","Bangalore","Store-03","Tablet","2024-01-03",24000),
("T007","East","Kolkata","Store-05","Laptop","2024-01-01",72000),
("T008","East","Kolkata","Store-05","Mobile","2024-01-02",28000),
("T009","East","Patna","Store-06","Tablet","2024-01-03",23000),
("T010","West","Mumbai","Store-07","Laptop","2024-01-01",80000),
("T011","West","Mumbai","Store-07","Mobile","2024-01-02",35000),
("T012","West","Pune","Store-08","Tablet","2024-01-03",27000),
("T013","North","Delhi","Store-01","Laptop","2024-01-04",76000),
("T014","South","Chennai","Store-04","Laptop","2024-01-04",79000),
("T015","East","Patna","Store-06","Mobile","2024-01-04",29000),
("T016","West","Pune","Store-08","Laptop","2024-01-04",77000),
("T017","North","Chandigarh","Store-02","Mobile","2024-01-05",31000),
("T018","South","Bangalore","Store-03","Mobile","2024-01-05",34000),
("T019","East","Kolkata","Store-05","Tablet","2024-01-05",25000),
("T020","West","Mumbai","Store-07","Tablet","2024-01-05",29000),
("T021","North","Delhi","Store-01","Tablet","2024-01-06",28000),
("T022","South","Chennai","Store-04","Tablet","2024-01-06",26000),
("T023","East","Patna","Store-06","Laptop","2024-01-06",74000),
("T024","West","Pune","Store-08","Mobile","2024-01-06",33000)
]
columns = [
"txn_id","region","city","store_id",
"product","sale_date","amount"
]
df_sales = spark.createDataFrame(sales_data, columns)
df_sales.show(5)
df_sales.printSchema()

+------+------+----------+--------+-------+----------+------+
|txn_id|region|      city|store_id|product| sale_date|amount|
+------+------+----------+--------+-------+----------+------+
|  T001| North|     Delhi|Store-01| Laptop|2024-01-01| 75000|
|  T002| North|     Delhi|Store-01| Mobile|2024-01-02| 32000|
|  T003| North|Chandigarh|Store-02| Tablet|2024-01-03| 26000|
|  T004| South| Bangalore|Store-03| Laptop|2024-01-01| 78000|
|  T005| South|   Chennai|Store-04| Mobile|2024-01-02| 30000|
+------+------+----------+--------+-------+----------+------+
only showing top 5 rows
root
 |-- txn_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- city: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- sale_date: string (nullable = true)
 |-- amount: long (nullable = true)



In [3]:
from pyspark.sql.functions import col,year
df_sales.select("txn_id","region","product","amount")

DataFrame[txn_id: string, region: string, product: string, amount: bigint]

In [6]:
df_sales.withColumnRenamed("amount","revenue")

DataFrame[txn_id: string, region: string, city: string, store_id: string, product: string, sale_date: string, revenue: bigint]

In [8]:
df_sales.withColumn("amount_in_thousands",col("amount") / 1000)

DataFrame[txn_id: string, region: string, city: string, store_id: string, product: string, sale_date: string, amount: bigint, amount_in_thousands: double]

In [18]:
df_sales.select("region","product").distinct()

DataFrame[region: string, product: string]

In [11]:
df_sales.drop("store_id")

DataFrame[txn_id: string, region: string, city: string, product: string, sale_date: string, amount: bigint]

In [12]:
df_sales.withColumn("sale_year",year("sale_date"))

DataFrame[txn_id: string, region: string, city: string, store_id: string, product: string, sale_date: string, amount: bigint, sale_year: int]

In [13]:
df_sales.select("txn_id","sale_date","region","city","product","amount")

DataFrame[txn_id: string, sale_date: string, region: string, city: string, product: string, amount: bigint]

In [14]:
from pyspark.sql.functions import col
df_sales.filter(col("amount")>50000)

DataFrame[txn_id: string, region: string, city: string, store_id: string, product: string, sale_date: string, amount: bigint]

In [15]:
df_sales.filter(col("product")=="Laptop")

DataFrame[txn_id: string, region: string, city: string, store_id: string, product: string, sale_date: string, amount: bigint]

In [16]:
df_sales.filter(col("region").isin("North","South"))

DataFrame[txn_id: string, region: string, city: string, store_id: string, product: string, sale_date: string, amount: bigint]

In [19]:
df_sales.filter(col("amount").between(25000, 75000))

DataFrame[txn_id: string, region: string, city: string, store_id: string, product: string, sale_date: string, amount: bigint]

In [20]:
df_sales.filter(col("city")=="Delhi")

DataFrame[txn_id: string, region: string, city: string, store_id: string, product: string, sale_date: string, amount: bigint]

In [21]:
df_sales.filter((col("region")=="North") & (col("amount")>50000))

DataFrame[txn_id: string, region: string, city: string, store_id: string, product: string, sale_date: string, amount: bigint]

In [22]:
df_sales.filter(col("amount")>50000).filter(col("region")=="North").explain(True)

== Parsed Logical Plan ==
'Filter '`=`('region, North)
+- Filter (amount#6L > cast(50000 as bigint))
   +- LogicalRDD [txn_id#0, region#1, city#2, store_id#3, product#4, sale_date#5, amount#6L], false

== Analyzed Logical Plan ==
txn_id: string, region: string, city: string, store_id: string, product: string, sale_date: string, amount: bigint
Filter (region#1 = North)
+- Filter (amount#6L > cast(50000 as bigint))
   +- LogicalRDD [txn_id#0, region#1, city#2, store_id#3, product#4, sale_date#5, amount#6L], false

== Optimized Logical Plan ==
Filter ((isnotnull(amount#6L) AND isnotnull(region#1)) AND ((amount#6L > 50000) AND (region#1 = North)))
+- LogicalRDD [txn_id#0, region#1, city#2, store_id#3, product#4, sale_date#5, amount#6L], false

== Physical Plan ==
*(1) Filter ((isnotnull(amount#6L) AND isnotnull(region#1)) AND ((amount#6L > 50000) AND (region#1 = North)))
+- *(1) Scan ExistingRDD[txn_id#0,region#1,city#2,store_id#3,product#4,sale_date#5,amount#6L]



In [23]:
from pyspark.sql.functions import sum,avg,max,min,count
df_sales.groupBy("region").agg(sum("amount").alias("total_sales"))

DataFrame[region: string, total_sales: bigint]

In [24]:
df_sales.groupBy("product").agg(avg("amount").alias("avg_sales"))

DataFrame[product: string, avg_sales: double]

In [25]:
df_sales.groupBy("city").agg(max("amount").alias("max_sales"))

DataFrame[city: string, max_sales: bigint]

In [27]:
df_sales.groupBy("region").agg(count("*").alias("txn_count"))

DataFrame[region: string, txn_count: bigint]

In [28]:
df_sales.groupBy("store_id").agg(sum("amount").alias("revenue"))

DataFrame[store_id: string, revenue: bigint]

In [29]:
df_sales.groupBy("region","product").count()

DataFrame[region: string, product: string, count: bigint]

In [30]:
df_sales.groupBy("city").agg(avg("amount").alias("avg_txn_value"))

DataFrame[city: string, avg_txn_value: double]

In [31]:
df_sales.groupBy("region")\
  .agg(sum("amount").alias("total_sales"))\
  .filter(col("total_sales")>200000)

DataFrame[region: string, total_sales: bigint]

In [33]:
df_sales.groupBy("region")\
  .agg(sum("amount").alias("total_sales"))\
  .filter(col("total_sales")>200000)

DataFrame[region: string, total_sales: bigint]

In [35]:
df_sales.groupBy("region").sum("amount").explain(True)

== Parsed Logical Plan ==
'Aggregate ['region], ['region, unresolvedalias('sum(amount#6L))]
+- LogicalRDD [txn_id#0, region#1, city#2, store_id#3, product#4, sale_date#5, amount#6L], false

== Analyzed Logical Plan ==
region: string, sum(amount): bigint
Aggregate [region#1], [region#1, sum(amount#6L) AS sum(amount)#121L]
+- LogicalRDD [txn_id#0, region#1, city#2, store_id#3, product#4, sale_date#5, amount#6L], false

== Optimized Logical Plan ==
Aggregate [region#1], [region#1, sum(amount#6L) AS sum(amount)#121L]
+- Project [region#1, amount#6L]
   +- LogicalRDD [txn_id#0, region#1, city#2, store_id#3, product#4, sale_date#5, amount#6L], false

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[region#1], functions=[sum(amount#6L)], output=[region#1, sum(amount)#121L])
   +- Exchange hashpartitioning(region#1, 200), ENSURE_REQUIREMENTS, [plan_id=39]
      +- HashAggregate(keys=[region#1], functions=[partial_sum(amount#6L)], output=[region#1, sum#123L])
     

In [36]:
from pyspark.sql.window import Window
from pyspark.sql.functions import sum, rank, dense_rank, row_number

w = Window.partitionBy("region").orderBy("sale_date") \
          .rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_sales.withColumn("running_total", sum("amount").over(w))


DataFrame[txn_id: string, region: string, city: string, store_id: string, product: string, sale_date: string, amount: bigint, running_total: bigint]

In [37]:
w = Window.partitionBy("store_id").orderBy(col("amount").desc())
df_sales.withColumn("row_num", row_number().over(w))

DataFrame[txn_id: string, region: string, city: string, store_id: string, product: string, sale_date: string, amount: bigint, row_num: int]

In [38]:
w = Window.partitionBy("region").orderBy(col("amount").desc())
df_sales.withColumn("dense_rank", dense_rank().over(w))

DataFrame[txn_id: string, region: string, city: string, store_id: string, product: string, sale_date: string, amount: bigint, dense_rank: int]

In [39]:
df_sales.withColumn("rank", rank().over(w)).filter(col("rank") <= 2)

DataFrame[txn_id: string, region: string, city: string, store_id: string, product: string, sale_date: string, amount: bigint, rank: int]

In [40]:
df_sales.select("amount",
          rank().over(w).alias("rank"),
          dense_rank().over(w).alias("dense_rank"))

DataFrame[amount: bigint, rank: int, dense_rank: int]

In [41]:
w = Window.partitionBy("store_id").orderBy("sale_date") \
          .rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_sales.withColumn("cumulative_sales", sum("amount").over(w))

DataFrame[txn_id: string, region: string, city: string, store_id: string, product: string, sale_date: string, amount: bigint, cumulative_sales: bigint]

In [42]:
w = Window.partitionBy("city").orderBy("sale_date")

df_sales.withColumn("first_txn", row_number().over(w)) \
  .withColumn("last_txn", row_number().over(w.orderBy(col("sale_date").desc())))

DataFrame[txn_id: string, region: string, city: string, store_id: string, product: string, sale_date: string, amount: bigint, first_txn: int, last_txn: int]

In [44]:
df_sales.select("region").explain(True)
df_sales.filter(col("amount") > 50000).explain(True)
df_sales.groupBy("region").sum("amount").explain(True)
df_sales.withColumn("rank", rank().over(w)).explain(True)

== Parsed Logical Plan ==
'Project ['region]
+- LogicalRDD [txn_id#0, region#1, city#2, store_id#3, product#4, sale_date#5, amount#6L], false

== Analyzed Logical Plan ==
region: string
Project [region#1]
+- LogicalRDD [txn_id#0, region#1, city#2, store_id#3, product#4, sale_date#5, amount#6L], false

== Optimized Logical Plan ==
Project [region#1]
+- LogicalRDD [txn_id#0, region#1, city#2, store_id#3, product#4, sale_date#5, amount#6L], false

== Physical Plan ==
*(1) Project [region#1]
+- *(1) Scan ExistingRDD[txn_id#0,region#1,city#2,store_id#3,product#4,sale_date#5,amount#6L]

== Parsed Logical Plan ==
'Filter '`>`('amount, 50000)
+- LogicalRDD [txn_id#0, region#1, city#2, store_id#3, product#4, sale_date#5, amount#6L], false

== Analyzed Logical Plan ==
txn_id: string, region: string, city: string, store_id: string, product: string, sale_date: string, amount: bigint
Filter (amount#6L > cast(50000 as bigint))
+- LogicalRDD [txn_id#0, region#1, city#2, store_id#3, product#4, sale_da